# Phase 2 — Step 2: the shared NER pipeline (tested with gold labels, no model yet)

Strict scoring means **one wrong character = a wrong entity**. So before training any model we build and test the
"plumbing" every experiment shares. All of it lives in **`src/ner_utils.py`**. Every Phase-2 notebook imports it, so the
experiments differ only in the model and the fine-tuning method (a fair comparison).

```
original text --normalise, keep a position map--> clean text --bpe32k--> tokens + character offsets
gold spans --> one BIO tag per token --> windows of <= 1536 tokens --> [model] --> tag scores per token
tag scores --merge windows--> Viterbi (valid BIO only) --> spans --map back--> ORIGINAL offsets --> scorer / prediction.json
```

This notebook tests every arrow, using the **gold labels in place of a model**:

1. Offset-preserving normalisation (Phase-1-style cleaning that cannot break offsets)
2. Gold → BIO tags → spans: the **ceiling** = best strict score any token-tagging model could reach
3. Windows: cutting documents into 1536-token pieces and merging them back; Viterbi
4. Where entities sit in a document (does a late window still see the party names from the header?)
5. Scorer sanity checks (known answers)
6. Submission writer (zip with `prediction.json` at the root)

Everything runs in about a minute, so nothing needs caching. A summary is saved to
`experiments/phase2/p2_02_pipeline_checks.json` for the report.


## 0. Setup

`importlib.reload(nu)` lets you edit `ner_utils.py` and re-run this cell without restarting the kernel.


In [1]:
import sys, json, time, importlib, tempfile
from pathlib import Path
from collections import Counter
import numpy as np
from tokenizers import Tokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ner_utils as nu
importlib.reload(nu)

data = nu.load_ner_data(PROJECT_ROOT)
LABELS = data["labels"]
TAGS, TAG2ID = nu.make_tags(LABELS)
ALLOWED, START_OK = nu.bio_constraints(TAGS)
tokenizer = Tokenizer.from_file(str(PROJECT_ROOT / "data" / "tokenizer" / "bpe32k.json"))
OUT = PROJECT_ROOT / "experiments" / "phase2"
OUT.mkdir(parents=True, exist_ok=True)
MAX_LEN, STRIDE = 1536, 1024
summary = {}

print(f"{len(TAGS)} tags: {TAGS[:5]} ...")
print("documents:", {f: len(t) for f, t in data["texts"].items()}, "| test:", len(data["test"]))

25 tags: ['O', 'B-APP', 'I-APP', 'B-RESP', 'I-RESP'] ...
documents: {'fold1': 35, 'fold2': 35} | test: 35


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Offset-preserving normalisation

Our Phase-1 models were trained on **cleaned** text (curly quotes → straight, soft hyphens deleted, `…` → `...`).
Raw Phase-2 text contains characters the models never saw in pretraining; a soft hyphen, for example, splits
`Parlia­ment` into odd byte pieces. But ordinary cleaning changes the length of the text and would shift every offset after it.

Solution: clean **one character at a time** and keep two position maps:

- `src[j]` = original position of cleaned character `j` (cleaned → original) — used for **predictions**
- `fwd[i]` = cleaned position of original character `i` (original → cleaned) — used for **gold labels**

A character can become 1 character (`“` → `"`), 3 characters (`…` → `...`) or 0 characters (soft hyphen).
Characters with no ASCII form (e.g. Devanagari) are kept, so no entity text is ever deleted.

**Check:** map every gold entity into the cleaned text and back. It must land on exactly the same offsets.


In [2]:
demo = "the “Parlia­mentary” Act… of 1950"
norm, src, fwd = nu.normalize_with_map(demo)
print("original  :", repr(demo), len(demo), "chars")
print("normalised:", repr(norm), len(norm), "chars")
a = norm.index("Parliamentary"); b = a + len("Parliamentary")
s, e = int(src[a]), int(src[b - 1]) + 1
print(f"'Parliamentary' = normalised {a}..{b}  ->  original {s}..{e} = {demo[s:e]!r}")

changed, n_chars, n_ok, bad = Counter(), 0, 0, []
for f in data["texts"]:
    for doc_id, text in data["texts"][f].items():
        norm, src, fwd = nu.normalize_with_map(text)
        n_chars += len(text)
        changed.update(ch for ch in text if nu.char_replacement(ch) != ch)
        for s, e, lab in data["golds"][f][doc_id]:
            a, b = fwd[s], fwd[e]
            back = (int(src[a]), int(src[b - 1]) + 1) if a < b else None
            if back == (s, e):
                n_ok += 1
            else:
                bad.append((f, doc_id, s, e, lab, text[s:e], back))
n_changed = sum(changed.values())
print(f"\ncharacters changed by normalisation: {n_changed:,} of {n_chars:,} ({n_changed / n_chars:.3%})")
print("most common:", [(c, nu.char_replacement(c), n) for c, n in changed.most_common(8)])
print(f"gold entities: {n_ok:,} map back exactly, {len(bad)} do not")
for x in bad[:5]:
    print("  BROKEN", x)
summary["normalisation"] = {"chars_changed": n_changed, "entities_ok": n_ok, "entities_broken": len(bad)}

original  : 'the “Parlia\xadmentary” Act… of 1950' 33 chars
normalised: 'the "Parliamentary" Act... of 1950' 34 chars
'Parliamentary' = normalised 5..18  ->  original 5..19 = 'Parlia\xadmentary'

characters changed by normalisation: 2,168 of 2,213,646 (0.098%)
most common: [('…', '...', 590), ('“', '"', 406), ('”', '"', 398), ('’', "'", 276), ('\xad', '', 271), ('‘', "'", 102), ('–', '-', 55), ('—', '-', 29)]
gold entities: 8,392 map back exactly, 1 do not
  BROKEN ('fold2', '108594766', 2954, 2967, 'DATE', '1,12,08,600/\xad', (2954, 2966))


## 2. Tokens, BIO tags and the round-trip ceiling

**BIO tagging:** every token gets one of 25 tags: `B-X` = first token of an X entity, `I-X` = a later token of the
same entity, `O` = not part of an entity. Example: `Jage Ram` → `B-APP I-APP`.
B is needed so that two neighbouring entities of the same type stay separate.

A token belongs to an entity if it shares at least one character with it. A byte-level BPE token such as `ĠState`
covers `" State"` (the space is part of the token), so we move each token's start past its leading space (`tok_start`).

We tokenise the documents **twice**: raw text and normalised text.


In [3]:
docs = {}
for normalize in (False, True):
    for f in data["texts"]:
        docs[(f, normalize)] = nu.encode_docs(data["texts"][f], tokenizer, normalize=normalize,
                                              desc=f"{f} normalize={normalize}")
test_docs = nu.encode_docs(data["test"], tokenizer, normalize=True, desc="test")

for normalize in (False, True):
    n_tok = sum(len(d.ids) for f in data["texts"] for d in docs[(f, normalize)])
    print(f"normalize={str(normalize):5s}: {n_tok:,} training tokens")

# example: 24 tokens around the first entity of the first fold-1 document
doc = docs[("fold1", True)][0]
tags = nu.spans_to_tags(doc, data["golds"]["fold1"][doc.doc_id], TAG2ID)
k0 = max(0, int(np.flatnonzero(tags)[0]) - 4)
print()
print("  ".join(f"{doc.norm[doc.tok_start[k]:doc.tok_end[k]]}/{TAGS[tags[k]]}" for k in range(k0, k0 + 24)))

normalize=False: 465,388 training tokens
normalize=True : 459,465 training tokens

./O  ../O  Appellants/O  Versus/O  STATE/B-RESP  OF/I-RESP  HARYANA/I-RESP  ../O  Respondent/O  J/O  U/O  D/O  G/O  M/O  E/O  N/O  T/O  R/O  ./O  BANUMATHI/O  ,/O  J/O  ./O  This/O


**Ceiling:** gold spans → tags → spans again, scored against the gold. A _perfect_ token-tagging model would get
exactly this score; whatever is lost here is lost for **every** model. Losses come from gold boundaries that fall
**inside** a token (Step 1: 152 entities, 1.81%), e.g. gold `... & Ors.` followed by `,` when the tokenizer
made the single token `.,` — the token is either in the entity (one character too many) or out (one too few).


In [4]:
gold_all = {**data["golds"]["fold1"], **data["golds"]["fold2"]}
roundtrip = {}
summary["ceiling"] = {}
for normalize in (False, True):
    pred_all = {}
    for f in data["texts"]:
        pred = {d.doc_id: nu.tags_to_spans(d, nu.spans_to_tags(d, data["golds"][f][d.doc_id], TAG2ID), TAGS)
                for d in docs[(f, normalize)]}
        sc = nu.strict_scores(data["golds"][f], pred, LABELS)
        lost = sum(v["fn"] for v in sc["per_label"].values())
        print(f"{f}  normalize={str(normalize):5s}  ceiling macro-F1 {sc['macro_f1']:.4f}  micro-F1 {sc['micro_f1']:.4f}  entities lost {lost}")
        summary["ceiling"][f"{f}_normalize={normalize}"] = {"macro_f1": sc["macro_f1"], "micro_f1": sc["micro_f1"], "lost": lost}
        pred_all.update(pred)
    roundtrip[normalize] = pred_all

sc = nu.strict_scores(gold_all, roundtrip[True], LABELS)
nu.print_scores(sc, "\nceiling per label, both folds, normalised text:")
summary["ceiling_per_label"] = {lab: round(v["r"], 4) for lab, v in sc["per_label"].items()}

fold1  normalize=False  ceiling macro-F1 0.9943  micro-F1 0.9927  entities lost 30
fold2  normalize=False  ceiling macro-F1 0.9924  micro-F1 0.9909  entities lost 39
fold1  normalize=True   ceiling macro-F1 0.9943  micro-F1 0.9927  entities lost 30
fold2  normalize=True   ceiling macro-F1 0.9923  micro-F1 0.9906  entities lost 40

ceiling per label, both folds, normalised text:
  macro-F1 0.9932 | macro-P 0.9932 | macro-R 0.9932 | F1(of macro P,R) 0.9932 | micro-F1 0.9917
  label           P      R     F1    TP    FP    FN  gold
  APP         0.985  0.985  0.985   401     6     6   407
  RESP        0.984  0.984  0.984   375     6     6   381
  A.COUNSEL   1.000  1.000  1.000   195     0     0   195
  R.COUNSEL   1.000  1.000  1.000   184     0     0   184
  JUDGE       1.000  1.000  1.000   245     0     0   245
  WIT         1.000  1.000  1.000   187     0     0   187
  AUTH        0.991  0.991  0.991   994     9     9  1003
  COURT       0.982  0.982  0.982   995    18    18  1013
 

**Which boundaries are lost, and how?** For every lost gold entity we compare it with the round-trip span:
`end +','` means the round trip added a `,` at the end. If one pattern dominates, a simple fix in span decoding
(e.g. trim a trailing `,`) could win back part of the ceiling. We look at this before deciding whether it is worth doing.


In [5]:
def context(text, s, e, w=25):
    return f"{text[max(0, s - w):s]}[[{text[s:e]}]]{text[e:e + w]}"

patterns, lost_by_label, examples = Counter(), Counter(), []
for f in data["texts"]:
    for d in docs[(f, True)]:
        g = set(data["golds"][f][d.doc_id])
        p = set(roundtrip[True][d.doc_id])
        for s, e, lab in sorted(g - p):
            lost_by_label[lab] += 1
            got = [x for x in p if x[0] < e and s < x[1]]
            if len(got) != 1:
                patterns["no single matching span"] += 1
                continue
            ps, pe, _ = got[0]
            parts = []
            if ps != s:
                parts.append(f"start {'+' if ps < s else '-'}{d.text[min(s, ps):max(s, ps)]!r}")
            if pe != e:
                parts.append(f"end {'+' if pe > e else '-'}{d.text[min(e, pe):max(e, pe)]!r}")
            patterns[", ".join(parts)] += 1
            if len(examples) < 12:
                examples.append((lab, context(d.text, s, e), d.text[ps:pe]))
print("lost entities per label:", dict(lost_by_label.most_common()))
print("\nmost common patterns (round trip vs gold):")
for pat, n in patterns.most_common(12):
    print(f"  {n:4d}  {pat}")
print("\nexamples  (gold in [[ ]]  ->  what the round trip gives):")
for lab, ctx, got in examples:
    print(f"  {lab:9s} {ctx!r}  ->  {got!r}")
summary["lost_patterns"] = dict(patterns.most_common(12))

lost entities per label: {'STAT': 19, 'COURT': 18, 'AUTH': 9, 'PREC': 9, 'APP': 6, 'RESP': 6, 'DATE': 3}

most common patterns (round trip vs gold):
    27  end +'s'
    16  end -' '
     9  end +'al'
     4  end +'h'
     4  end +'"'
     2  end +')'
     2  end +']'
     2  end -','
     1  end +'ary'
     1  end +'e'
     1  start +'I'
     1  end -'\xad'

examples  (gold in [[ ]]  ->  what the round trip gives):
  AUTH      '7.9.1997 struck down the [[Railway Board]]s instructions on the bas'  ->  'Railway Boards'
  AUTH      'these, implementation of [[Railway Board]]s decision to enhance the'  ->  'Railway Boards'
  STAT      'institutions". After the [[Constitution]]al Amendment (42 Amendmen'  ->  'Constitutional'
  AUTH      ' agencies, including the [[State Government]]s and the Universities co'  ->  'State Governments'
  AUTH      'taining the views of the [[State Government]]s and the Universities. N'  ->  'State Governments'
  COURT     'he nomenclature) and the [[High Cour

## 3. Windows and Viterbi

Our Phase-1 models were trained on 1536 positions (RoPE); positions beyond that were never seen, so we never feed
more than 1536 tokens. Every training document is longer (median 5,625 tokens), so each document is cut into
**overlapping windows**: length 1536, a new window every 1024 tokens (overlap 512). The last window is aligned to the
end of the document, so all windows are full length (no padding).

At prediction time a token can sit in two windows. We take its scores from the window where it is **farthest from an
edge** (most context on both sides). The overlap (512) is much longer than the longest entity (51 tokens), so every
entity lies completely inside at least one window.

**Check:** feed the merge step "perfect" scores (log-probability 0 for the gold tag, −20 for all others), window by
window. After merging and Viterbi we must get exactly the gold tags back, for every document.


In [6]:
n_same = n_docs = 0
n_windows = Counter()
longest = 0
t0 = time.time()
for f in data["texts"]:
    for d in docs[(f, True)]:
        tags = nu.spans_to_tags(d, data["golds"][f][d.doc_id], TAG2ID)
        n = len(tags)
        perfect = np.full((n, len(TAGS)), -20.0, dtype=np.float32)
        perfect[np.arange(n), tags] = 0.0
        windows = nu.make_windows(n, MAX_LEN, STRIDE)
        merged = nu.merge_windows(n, windows, [perfect[a:b] for a, b in windows])
        path = nu.viterbi(merged, ALLOWED, START_OK)
        n_same += int(np.array_equal(path, tags))
        n_docs += 1
        n_windows[f] += len(windows)
        for s, e, lab in data["golds"][f][d.doc_id]:       # longest entity in tokens
            k = np.flatnonzero((d.tok_start < d.fwd[e]) & (d.tok_end > d.fwd[s]))
            longest = max(longest, len(k))
secs = (time.time() - t0) / n_docs
n_windows["test"] = sum(len(nu.make_windows(len(d.ids), MAX_LEN, STRIDE)) for d in test_docs)
print(f"windows + merge + Viterbi reproduce the gold tags exactly: {n_same} of {n_docs} documents")
print(f"windows of {MAX_LEN} tokens, stride {STRIDE}: {dict(n_windows)}")
print(f"longest entity: {longest} tokens (overlap is {MAX_LEN - STRIDE})")
print(f"merge + Viterbi time: {secs:.2f} s per document")
summary["windows"] = {"max_len": MAX_LEN, "stride": STRIDE, "counts": dict(n_windows), "reproduced": f"{n_same}/{n_docs}"}

windows + merge + Viterbi reproduce the gold tags exactly: 70 of 70 documents
windows of 1536 tokens, stride 1024: {'fold1': 223, 'fold2': 223, 'test': 240}
longest entity: 51 tokens (overlap is 512)
merge + Viterbi time: 0.04 s per document


**Why Viterbi and not just the best tag per token?** Taking the best tag per token (argmax) can produce invalid
sequences like `O I-APP I-APP` (an entity without a beginning). Viterbi finds the sequence with the highest total
score **among valid sequences only** (`I-X` may only follow `B-X` or `I-X`). Tiny example with 3 tokens:


In [7]:
toy_tags = ["O", "B-APP", "I-APP"]
toy_allowed, toy_start = nu.bio_constraints(toy_tags)
toy = np.log(np.array([[0.90, 0.05, 0.05],     # token 1: clearly O
                       [0.10, 0.40, 0.50],     # token 2: I-APP slightly ahead of B-APP
                       [0.10, 0.10, 0.80]]))   # token 3: I-APP
print("argmax :", [toy_tags[i] for i in toy.argmax(1)], " <- invalid: I-APP right after O")
print("Viterbi:", [toy_tags[i] for i in nu.viterbi(toy, toy_allowed, toy_start)])

argmax : ['O', 'I-APP', 'I-APP']  <- invalid: I-APP right after O
Viterbi: ['O', 'B-APP', 'I-APP']


## 4. Where do entities sit in a document?

Parties (APP, RESP), counsel and judges are introduced in the **header** at the top of a judgment. Later the text
may say "the appellant Jage Ram". A window from the middle of a document no longer contains the header, so the model
must decide APP vs RESP vs WIT from nearby words only.

For each label: the median token position, the share of entities **outside the first window** (token ≥ 1536), and,
among those, the share whose exact text **also appears in the first window**. A high last number means extra context
from the header (e.g. adding the header to every window) could help that label.


In [8]:
rows = []
for f in data["texts"]:
    for d in docs[(f, True)]:
        last = min(MAX_LEN, len(d.ids)) - 1
        header = d.text[: int(d.src[d.tok_end[last] - 1]) + 1]       # original text covered by the first window
        for s, e, lab in data["golds"][f][d.doc_id]:
            k = int(np.searchsorted(d.tok_end, d.fwd[s], side="right"))  # first token of the entity
            rows.append((lab, k, k >= MAX_LEN, d.text[s:e] in header))
print(f"{'label':10s} {'n':>5s} {'median tok':>10s} {'outside 1st window':>19s} {'... text also in 1st window':>28s}")
summary["position"] = {}
for lab in LABELS:
    r = [x for x in rows if x[0] == lab]
    out = [x for x in r if x[2]]
    share_out = len(out) / len(r)
    share_hdr = sum(x[3] for x in out) / len(out) if out else 0.0
    print(f"{lab:10s} {len(r):5d} {int(np.median([x[1] for x in r])):10d} {share_out:19.1%} {share_hdr:28.1%}")
    summary["position"][lab] = {"outside_first_window": round(share_out, 3), "of_those_in_header": round(share_hdr, 3)}

label          n median tok  outside 1st window  ... text also in 1st window
APP          407       1053               39.3%                        83.1%
RESP         381       1436               49.1%                        87.7%
A.COUNSEL    195       1318               40.5%                        12.7%
R.COUNSEL    184       1926               53.8%                        20.2%
JUDGE        245       1540               50.2%                        28.5%
WIT          187       1573               50.3%                        67.0%
AUTH        1003       3393               78.0%                        70.1%
COURT       1013       2829               69.9%                        73.9%
STAT        1260       3398               76.4%                        53.5%
PREC        1183       3554               87.8%                        15.9%
DATE        1575       1720               54.0%                        33.3%
CASENO       760       1472               49.2%                        48.9%

## 5. Scorer sanity checks

Each line has a known correct answer. If all match, the scorer is trustworthy.
Macro-F1 = average of the 12 per-label F1 scores, so removing **one whole label** costs exactly 1/12.


In [9]:
g = data["golds"]["fold2"]
def edit(fn):
    return {d: [x for x in (fn(s, e, l) for s, e, l in spans) if x] for d, spans in g.items()}

swap = {"APP": "RESP", "RESP": "APP"}
checks = {
    "gold vs gold                  (expect 1.0000)": g,
    "every span shifted +1 char    (expect 0.0000)": edit(lambda s, e, l: (s + 1, e + 1, l)),
    "only the end shortened by 1   (expect 0.0000)": edit(lambda s, e, l: (s, e - 1, l)),
    "all DATE removed       (expect 11/12 = 0.9167)": edit(lambda s, e, l: None if l == "DATE" else (s, e, l)),
    "APP <-> RESP swapped   (expect 10/12 = 0.8333)": edit(lambda s, e, l: (s, e, swap.get(l, l))),
    "nothing predicted             (expect 0.0000)": {d: [] for d in g},
}
for name, pred in checks.items():
    print(f"{name} -> macro-F1 {nu.strict_scores(g, pred, LABELS)['macro_f1']:.4f}")

cats, confusion, _ = nu.error_breakdown(g, checks["APP <-> RESP swapped   (expect 10/12 = 0.8333)"])
print("\nerror breakdown of the swap:", dict(cats), "| confusion (gold, predicted):", dict(confusion))

gold vs gold                  (expect 1.0000) -> macro-F1 1.0000
every span shifted +1 char    (expect 0.0000) -> macro-F1 0.0000
only the end shortened by 1   (expect 0.0000) -> macro-F1 0.0000
all DATE removed       (expect 11/12 = 0.9167) -> macro-F1 0.9167
APP <-> RESP swapped   (expect 10/12 = 0.8333) -> macro-F1 0.8333
nothing predicted             (expect 0.0000) -> macro-F1 0.0000

error breakdown of the swap: {'right span, wrong label': 390} | confusion (gold, predicted): {('RESP', 'APP'): 190, ('APP', 'RESP'): 200}


## 6. Submission writer

`write_submission` writes `prediction.json` with **every** test document (the empty document `1305957` gets `[]`),
zips it with `prediction.json` at the zip root, then re-opens the zip and validates it as the grader would.

- Test 1: fold-2 gold written as if it were a submission, read back, scored → must be 1.0 (tests JSON write/read).
- Test 2: empty predictions for the test set → format check only (**never upload this**). Written to a temporary folder.


In [10]:
with tempfile.TemporaryDirectory() as tmp:
    zp = nu.write_submission(data["golds"]["fold2"], data["texts"]["fold2"], LABELS, Path(tmp) / "gold_fold2")
    back = nu.check_submission(zp, data["texts"]["fold2"], LABELS, verbose=False)
    print("fold-2 gold -> zip -> read back, macro-F1:", nu.strict_scores(data["golds"]["fold2"], back, LABELS)["macro_f1"])
    nu.write_submission({d: [] for d in data["test"]}, data["test"], LABELS, Path(tmp) / "empty_test")
print("test documents with 0 tokens:", [d.doc_id for d in test_docs if len(d.ids) == 0])

submission OK: C:\Users\Atharva\AppData\Local\Temp\tmp6qr9xvwl\gold_fold2\submission.zip | 35 documents | 4,273 entities
fold-2 gold -> zip -> read back, macro-F1: 1.0
submission OK: C:\Users\Atharva\AppData\Local\Temp\tmp6qr9xvwl\empty_test\submission.zip | 35 documents | 0 entities
test documents with 0 tokens: ['1305957']


## 7. Save the summary (for the report)


In [11]:
with open(OUT / "p2_02_pipeline_checks.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("saved", OUT / "p2_02_pipeline_checks.json")

saved c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\experiments\phase2\p2_02_pipeline_checks.json


## What to paste back

The outputs of sections 1–6.
